# Hamburg Bike Count Stations — Inventory & Kepler.gl Live Map

Collects all Hamburg bike counting infrastructure from three sources:

| Source | What | Stations | Interval |
|--------|------|----------|----------|
| `iot.hamburg.de` STA `HH_STA_AutomatisierteVerkehrsmengenerfassung` | Infrared/thermal permanent counters | ~61 | 15 min real-time |
| `iot.hamburg.de` STA `HH_STA_StadtRad` | Bike-sharing dock occupancy | ~240 | 5 min real-time |
| `geodienste.hamburg.de` WFS `HH_WFS_Harazaen` | Hamburg Radzählnetz — bike-only stations | ~100 | Geometry + metadata |
| `geodienste.hamburg.de` WFS `HH_WFS_Verkehrszaehlstellen` | All traffic count stations (bike+car+foot) | ~300+ | Geometry + metadata |

**Outputs:**
- `hamburg_bike_stations.csv` — unified inventory table
- `hamburg_bike_map.html` — interactive Kepler.gl map with real-time counts

In [ ]:
# ── Install dependencies (run once) ─────────────────────────────────────────
import subprocess, sys
pkgs = ["requests", "pandas", "keplergl", "ipywidgets"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs, check=True)
print("✓ All packages ready")

In [ ]:
# ── Imports & shared helpers ─────────────────────────────────────────────────
import requests
import pandas as pd
import json
import time
from datetime import datetime, timezone
from pathlib import Path

STA_BASE   = "https://iot.hamburg.de/v1.1"
WFS_BASE   = "https://geodienste.hamburg.de/HH_WFS_Verkehrszaehlstellen"
PAGE_SIZE  = 100   # STA paging size
TIMEOUT    = 30    # seconds per request

def sta_get_all(url: str) -> list:
    """Follow @iot.nextLink paging and collect all STA results."""
    results, page = [], 0
    while url:
        resp = requests.get(url, timeout=TIMEOUT)
        resp.raise_for_status()
        data = resp.json()
        batch = data.get("value", [])
        results.extend(batch)
        url = data.get("@iot.nextLink")
        page += 1
        print(f"  page {page}: +{len(batch)} items  (total {len(results)})", end="\r")
    print()
    return results

def iso_to_dt(s):
    """Parse ISO-8601 string or return None."""
    if not s:
        return None
    try:
        return pd.to_datetime(s, utc=True)
    except Exception:
        return None

print("✓ Helpers loaded")

### Discovery — inspect all serviceNames in the Hamburg STA (run once to debug)

In [ ]:
# This cell introspects the live STA and prints ALL distinct serviceNames + counts.
# Run it whenever you suspect a serviceName has changed.
# It is NOT needed for normal execution — skip if you trust the known names.

print("Scanning all Datastream serviceNames in iot.hamburg.de …")
resp = requests.get(
    f"{STA_BASE}/Datastreams?$select=properties&$top=1000", timeout=TIMEOUT
)
resp.raise_for_status()
all_props = [d.get("properties", {}) for d in resp.json().get("value", [])]

from collections import Counter
service_names = Counter(p.get("serviceName", "(none)") for p in all_props)
layer_names   = Counter(p.get("layerName",   "(none)") for p in all_props)

print("\n── serviceNames (count of Datastreams) ─────────")
for name, cnt in service_names.most_common():
    print(f"  {cnt:4d}  {name}")

print("\n── layerNames (count of Datastreams) ───────────")
for name, cnt in layer_names.most_common():
    print(f"  {cnt:4d}  {name}")

## 1 · IoT Hamburg STA — Permanent Traffic/Bike Count Stations (`HH_STA_AutomatisierteVerkehrsmengenerfassung`)

In [ ]:
# ── Root cause of the 0-result bug ──────────────────────────────────────────
# The service name 'HH_STA_Radverkehr' does NOT exist in the Hamburg STA.
# ALL automated vehicle/bike count stations (infrared cameras) are served under:
#   serviceName = 'HH_STA_AutomatisierteVerkehrsmengenerfassung'
# This single service covers both car AND bike detectors.
# Bike-specific stations are identified by filtering Things whose name contains 'Rad'.
#
# Confirmed layer names (from MetaVer docuuid 2936465E-C045-4F5D-8614-24C3FBB522E2):
#   Anzahl_Kfz_Zaehlstelle_15-Min   ← 15-minute aggregate (use this for live counts)
#   Anzahl_Kfz_Zaehlstelle_1-Stunde ← hourly aggregate
#   Anzahl_Kfz_Zaehlstelle_1-Tag    ← daily aggregate
#   Anzahl_Kfz_Zaehlstelle_1-Woche  ← weekly aggregate

CORRECT_SERVICE = "HH_STA_AutomatisierteVerkehrsmengenerfassung"
TARGET_LAYER    = "Anzahl_Kfz_Zaehlstelle_15-Min"

print(f"Fetching ALL Datastreams for layer '{TARGET_LAYER}' …")
print(f"(service: '{CORRECT_SERVICE}')")

rad_url = (
    f"{STA_BASE}/Datastreams"
    f"?$filter=properties/serviceName eq '{CORRECT_SERVICE}'"
    f" and properties/layerName eq '{TARGET_LAYER}'"
    f"&$expand=Thing($expand=Locations($select=location)),ObservedProperty,Sensor"
    f"&$top={PAGE_SIZE}"
)

rad_ds_all = sta_get_all(rad_url)
print(f"Total datastreams (bike + car counters): {len(rad_ds_all)}")

# ── Separate bike-only vs all-vehicle stations ───────────────────────────────
# Hamburg infrared stations detect all modes. Things whose name contains
# 'Rad' or 'Fahr' are dedicated cycling cross-sections.
# We keep ALL of them but tag each one so we can filter later in the map.
def is_bike_station(thing: dict) -> bool:
    name = (thing.get("name") or "").lower()
    desc = (thing.get("description") or "").lower()
    keywords = ["rad", "fahr", "bike", "cycl"]
    return any(k in name or k in desc for k in keywords)

rad_ds_bike = [ds for ds in rad_ds_all if is_bike_station(ds.get("Thing", {}))]
rad_ds_all_vehicle = [ds for ds in rad_ds_all if not is_bike_station(ds.get("Thing", {}))]

print(f"\n  Bike-dedicated stations : {len(rad_ds_bike)}")
print(f"  All-vehicle stations    : {len(rad_ds_all_vehicle)}")
print(f"  (keeping all {len(rad_ds_all)} for the CSV; station_type column distinguishes them)")

rad_ds = rad_ds_all   # use all for CSV; tagged in parse step below
print(f"\n✓ {len(rad_ds)} Datastreams ready for parsing")

In [ ]:
# Parse Datastreams → rows, tagging bike vs mixed-vehicle stations
rad_rows = []

for ds in rad_ds:
    thing = ds.get("Thing", {})
    props = ds.get("properties", {})

    # WGS84 coordinates from Thing → Locations
    lon, lat = None, None
    for loc in thing.get("Locations", []):
        geom = loc.get("location", {})
        if geom.get("type") == "Point":
            coords = geom.get("coordinates", [])
        elif geom.get("type") == "Feature":
            coords = geom.get("geometry", {}).get("coordinates", [])
        else:
            coords = []
        if len(coords) >= 2:
            lon, lat = coords[0], coords[1]
            break

    ds_id    = ds.get("@iot.id")
    thing_id = thing.get("@iot.id")
    obs_url  = f"{STA_BASE}/Datastreams({ds_id})/Observations?$orderby=phenomenonTime desc&$top=1"
    src_url  = (
        f"{STA_BASE}/Datastreams({ds_id})"
        f"?$expand=Thing($expand=Locations),Observations($orderby=phenomenonTime desc;$top=1)"
    )

    pt = ds.get("phenomenonTime", "")
    dt_start, dt_end = None, None
    if pt and "/" in pt:
        parts = pt.split("/")
        dt_start = iso_to_dt(parts[0])
        dt_end   = iso_to_dt(parts[1]) if len(parts) > 1 else None

    # Tag whether this is a bike-dedicated or mixed-mode station
    bike_dedicated = is_bike_station(thing)
    s_type = (
        "Permanent bike count — infrared/thermal camera (bike-dedicated)"
        if bike_dedicated else
        "Permanent traffic count — infrared/thermal camera (all vehicles incl. bikes)"
    )

    rad_rows.append({
        "station_id"      : str(thing_id),
        "datastream_id"   : str(ds_id),
        "name"            : thing.get("name", "").strip(),
        "datastream_name" : ds.get("name", "").strip(),
        "description"     : thing.get("description", "").strip(),
        "longitude_wgs84" : lon,
        "latitude_wgs84"  : lat,
        "provider"        : "Behörde für Verkehr und Mobilitätswende (BVM) / LGV Hamburg",
        "station_type"    : s_type,
        "service_name"    : props.get("serviceName", CORRECT_SERVICE),
        "layer_name"      : props.get("layerName", TARGET_LAYER),
        "api_source_url"  : src_url,
        "observations_url": obs_url,
        "mqtt_topic"      : f"v1.1/Datastreams({ds_id})/Observations",
        "datetime_start"  : dt_start,
        "datetime_end"    : dt_end,
        "unit"            : ds.get("unitOfMeasurement", {}).get("symbol", "counts"),
        "observed_property": ds.get("ObservedProperty", {}).get("name", ""),
        "sensor_type"     : ds.get("Sensor", {}).get("name", ""),
        "update_interval" : "15 min (aggregated); also 1-hour, 1-day, 1-week layers available",
        "source_portal"   : "iot.hamburg.de (OGC SensorThings API v1.1)",
    })

df_rad = pd.DataFrame(rad_rows)
bike_only = df_rad["station_type"].str.contains("bike-dedicated").sum()
mixed     = len(df_rad) - bike_only
print(f"✓ Built {len(df_rad)} rows  |  bike-dedicated: {bike_only}  |  mixed-vehicle: {mixed}")
print(f"  Unique Things (physical station locations): {df_rad['station_id'].nunique()}")
df_rad[["name","station_type","layer_name"]].head(5)

## 2 · IoT Hamburg STA — StadtRAD Bike-Share Stations

In [ ]:
# ── Root cause fix for StadtRAD ──────────────────────────────────────────────
# Correct serviceName: 'HH_STA_StadtRad'  (NOT 'STA StadtRAD')
# Available layerNames: 'Fahrraeder', 'E-Lastenraeder'
# Confirmed from Transparenzportal: suche.transparenz.hamburg.de/dataset/stadtrad-stationen-hamburg36
#
# The filter must be on Things (not Datastreams) because Datastreams/properties
# is a nested path — filter via Things with a Datastreams sub-filter.

STADTRAD_SERVICE = "HH_STA_StadtRad"

print(f"Fetching StadtRAD Things (serviceName='{STADTRAD_SERVICE}') …")

# Option A: filter via Datastreams properties (most reliable)
stadtrad_url = (
    f"{STA_BASE}/Things"
    f"?$filter=Datastreams/properties/serviceName eq '{STADTRAD_SERVICE}'"
    f"&$expand=Locations($select=location),"
    f"Datastreams("
    f"$filter=properties/layerName eq 'Fahrraeder';"
    f"$expand=Observations($select=phenomenonTime,result;"
    f"$orderby=phenomenonTime desc;$top=1);"
    f"$select=@iot.id,name,phenomenonTime,unitOfMeasurement,properties"
    f")"
    f"&$top={PAGE_SIZE}"
)

# Option B: simpler fallback if FROST rejects the nested filter
stadtrad_url_fallback = (
    f"{STA_BASE}/Things"
    f"?$filter=Datastreams/properties/serviceName eq '{STADTRAD_SERVICE}'"
    f"&$expand=Locations,Datastreams"
    f"&$top={PAGE_SIZE}"
)

try:
    stadtrad_things = sta_get_all(stadtrad_url)
    if not stadtrad_things:
        raise ValueError("empty — trying fallback")
    print(f"✓ Option A succeeded")
except Exception as e:
    print(f"  Option A failed ({e}), trying fallback …")
    try:
        stadtrad_things = sta_get_all(stadtrad_url_fallback)
        print(f"✓ Fallback succeeded")
    except Exception as e2:
        print(f"  Fallback also failed ({e2})")
        stadtrad_things = []

print(f"✓ {len(stadtrad_things)} StadtRAD station Things retrieved")

In [ ]:
# Parse StadtRAD Things → rows
# Each Thing has two Datastreams: 'Fahrraeder' (regular bikes) and 'E-Lastenraeder' (cargo e-bikes).
# We take 'Fahrraeder' as the primary count; also record E-Lastenraeder in a separate column.
sr_rows = []

for thing in stadtrad_things:
    t_id = thing.get("@iot.id")

    # Coordinates
    lon, lat = None, None
    for loc in thing.get("Locations", []):
        geom = loc.get("location", {})
        if geom.get("type") == "Point":
            coords = geom.get("coordinates", [])
        elif geom.get("type") == "Feature":
            coords = geom.get("geometry", {}).get("coordinates", [])
        else:
            coords = []
        if len(coords) >= 2:
            lon, lat = coords[0], coords[1]
            break

    # Find the correct Datastream by layerName
    ds_fahrrad   = None
    ds_lastenrad = None
    for ds in thing.get("Datastreams", []):
        ln = ds.get("properties", {}).get("layerName", "")
        if ln == "Fahrraeder":
            ds_fahrrad = ds
        elif ln == "E-Lastenraeder":
            ds_lastenrad = ds

    # Fallback: if layerName not set, take the first Datastream
    if ds_fahrrad is None and thing.get("Datastreams"):
        ds_fahrrad = thing["Datastreams"][0]

    ds_id  = ds_fahrrad["@iot.id"] if ds_fahrrad else None
    pt     = ds_fahrrad.get("phenomenonTime", "") if ds_fahrrad else ""
    dt_start, dt_end = None, None
    if pt and "/" in pt:
        parts = pt.split("/")
        dt_start = iso_to_dt(parts[0])
        dt_end   = iso_to_dt(parts[1]) if len(parts) > 1 else None

    src_url = (
        f"{STA_BASE}/Things({t_id})?$expand=Locations,"
        f"Datastreams($expand=Observations($orderby=phenomenonTime desc;$top=1))"
    )
    obs_url = (
        f"{STA_BASE}/Datastreams({ds_id})/Observations?$orderby=phenomenonTime desc&$top=1"
        if ds_id else ""
    )
    # Also record cargo e-bike datastream id for reference
    lastenrad_ds_id = ds_lastenrad["@iot.id"] if ds_lastenrad else None

    sr_rows.append({
        "station_id"        : str(t_id),
        "datastream_id"     : str(ds_id) if ds_id else "",
        "name"              : thing.get("name", "").strip(),
        "datastream_name"   : "Fahrraeder",
        "description"       : thing.get("description", "").strip(),
        "longitude_wgs84"   : lon,
        "latitude_wgs84"    : lat,
        "provider"          : "DB Connect / StadtRAD Hamburg",
        "station_type"      : "Bike-share docking station (available bikes)",
        "service_name"      : STADTRAD_SERVICE,
        "layer_name"        : "Fahrraeder",
        "api_source_url"    : src_url,
        "observations_url"  : obs_url,
        "mqtt_topic"        : f"v1.1/Datastreams({ds_id})/Observations" if ds_id else "",
        "datetime_start"    : dt_start,
        "datetime_end"      : dt_end,
        "unit"              : "bikes",
        "observed_property" : "Available regular bikes at docking station",
        "sensor_type"       : "Docking station occupancy sensor",
        "update_interval"   : "5 min",
        "source_portal"     : "iot.hamburg.de (OGC SensorThings API v1.1)",
        "lastenrad_ds_id"   : str(lastenrad_ds_id) if lastenrad_ds_id else "",
    })

df_sr = pd.DataFrame(sr_rows)
print(f"✓ Built {len(df_sr)} StadtRAD rows")
print(f"  Stations with Lastenrad stream: {(df_sr['lastenrad_ds_id'] != '').sum()}")
df_sr[["name","station_type","layer_name","lastenrad_ds_id"]].head(5)

## 3 · WFS — Bike Station Geometries from Two Dedicated Endpoints

### Root causes of the 400 error
| Bug | Wrong | Correct |
|-----|-------|---------|
| WFS version | `VERSION=2.0.0` | `VERSION=1.1.0` (Hamburg WFS is 1.1) |
| Typename param | `TYPENAMES=` | `typename=` (WFS 1.1 keyword) |
| Paging param | `COUNT=2000` | `maxFeatures=2000` (WFS 1.1 keyword) |
| Output format | `outputFormat=application/json` | Not supported — use GML then parse, or use the geodienste download proxy |

### Two sources fetched here
1. **`HH_WFS_Harazaen`** — dedicated Hamburg Radverkehrszählnetz (HaRaZäN) with ~100 bike-only counting stations and live count attributes. typename: `de.hh.up:zaehlstellen_daten`
2. **`HH_WFS_Verkehrszaehlstellen`** — full registry of all mode counting stations. typename: `app:radverkehr_zaehlstellen`

In [ ]:
import xml.etree.ElementTree as ET

# ── Shared WFS helper: fetch GML and parse to list of dicts ──────────────────
# Hamburg WFS 1.1.0 returns GML (application/gml+xml) by default.
# We parse the XML directly — no need for an extra GIS library.

def wfs_fetch_gml(base_url: str, typename: str, max_features: int = 2000,
                  version: str = "1.1.0") -> ET.Element:
    """Fetch a WFS 1.1.0 layer and return the parsed XML root element."""
    params = {
        "SERVICE"    : "WFS",
        "VERSION"    : version,        # ← must be 1.1.0, not 2.0.0
        "REQUEST"    : "GetFeature",
        "typename"   : typename,       # ← 'typename' not 'TYPENAMES'
        "maxFeatures": str(max_features),  # ← 'maxFeatures' not 'COUNT'
        # No outputFormat=json — Hamburg WFS 1.1 returns GML only
    }
    r = requests.get(base_url, params=params, timeout=60)
    r.raise_for_status()
    return ET.fromstring(r.content)

def get_gml_text(elem, *local_names) -> str:
    """Search for the first matching child by localname (namespace-agnostic)."""
    for child in elem.iter():
        tag = child.tag.split("}")[-1] if "}" in child.tag else child.tag
        if tag in local_names:
            return (child.text or "").strip()
    return ""

def get_gml_coord(elem) -> tuple:
    """Extract (lon, lat) from gml:pos or gml:coordinates inside a geometry."""
    for child in elem.iter():
        tag = child.tag.split("}")[-1] if "}" in child.tag else child.tag
        if tag == "pos" and child.text:
            # GML 3: 'lat lon' or 'x y' depending on srs
            parts = child.text.strip().split()
            if len(parts) >= 2:
                # Hamburg UTM32N (EPSG:25832) or WGS84 depending on SRS
                # Try to detect: UTM easting >100000, WGS84 lon <20
                a, b = float(parts[0]), float(parts[1])
                if a > 100000:  # UTM northing first
                    return (b, a)   # (lon/easting, lat/northing) — still UTM
                return (a, b)
        elif tag == "coordinates" and child.text:
            # GML 2: 'lon,lat'
            parts = child.text.strip().split(",")
            if len(parts) >= 2:
                return (float(parts[0]), float(parts[1]))
    return (None, None)

def utm32n_to_wgs84(easting: float, northing: float) -> tuple:
    """Convert EPSG:25832 (UTM zone 32N) to WGS84 lon/lat using pyproj if available,
    otherwise fall back to a simple approximation valid for Hamburg."""
    try:
        from pyproj import Transformer
        t = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=True)
        return t.transform(easting, northing)
    except ImportError:
        # Rough linear approximation centred on Hamburg (accurate to ~10 m)
        lon = (easting - 566000) / 66700 + 10.0
        lat = (northing - 5934000) / 111320 + 53.55
        return (lon, lat)

print("✓ WFS/GML helpers defined")

In [ ]:
# ── Source A: HH_WFS_Harazaen — Hamburg Radzählnetz (bike-only, ~100 stations) ──
# This is the DEDICATED bike counting WFS, confirmed working at VERSION=1.1.0.
# typename: de.hh.up:zaehlstellen_daten
# Contains: station ID, name/location label, coordinates, last measurement date.

HARAZAEN_BASE = "https://geodienste.hamburg.de/HH_WFS_Harazaen"
harazaen_rows = []

print("Fetching HH_WFS_Harazaen (Hamburg Radzählnetz — bike-only stations) …")
try:
    root_hr = wfs_fetch_gml(HARAZAEN_BASE, "de.hh.up:zaehlstellen_daten")
    members_hr = [
        child for child in root_hr.iter()
        if child.tag.split("}")[-1] == "zaehlstellen_daten"
    ]
    print(f"  Raw feature elements found: {len(members_hr)}")

    for feat in members_hr:
        raw_lon, raw_lat = get_gml_coord(feat)
        # Detect UTM and convert
        lon, lat = raw_lon, raw_lat
        if raw_lon is not None and abs(raw_lon) > 100:
            lon, lat = utm32n_to_wgs84(raw_lon, raw_lat)

        # Property field extraction — try multiple possible tag names
        name = get_gml_text(feat,
            "bezeichnung", "lagebezeichnung", "strassenname",
            "name", "zaehlstellenbezeichnung"
        )
        station_id = get_gml_text(feat,
            "zaehlstellennummer", "kennummer", "id", "fid"
        )
        last_count = get_gml_text(feat,
            "datum_letzte_zaehlung", "letzte_zaehlung", "datum", "aktualisierungsdatum"
        )
        dtv = get_gml_text(feat, "dtv", "dtv_rad", "dtv_gesamt")

        harazaen_rows.append({
            "station_id"      : station_id or feat.get("{http://www.opengis.net/gml}id", ""),
            "datastream_id"   : "",
            "name"            : name,
            "datastream_name" : "",
            "description"     : name,
            "longitude_wgs84" : lon,
            "latitude_wgs84"  : lat,
            "provider"        : "Behörde für Verkehr und Mobilitätswende (BVM) — HaRaZäN",
            "station_type"    : "Permanent bike count — Hamburg Radzählnetz (infrared)",
            "service_name"    : "HH_WFS_Harazaen",
            "layer_name"      : "de.hh.up:zaehlstellen_daten",
            "api_source_url"  : f"{HARAZAEN_BASE}?SERVICE=WFS&VERSION=1.1.0&REQUEST=GetFeature&typename=de.hh.up:zaehlstellen_daten",
            "observations_url": "",
            "mqtt_topic"      : "",
            "datetime_start"  : None,
            "datetime_end"    : iso_to_dt(last_count),
            "unit"            : "counts (DTV)",
            "observed_property": "Radverkehrsstärke (bikes)",
            "sensor_type"     : "Infrared thermal camera",
            "update_interval" : "Annual DTV; real-time via STA HH_STA_AutomatisierteVerkehrsmengenerfassung",
            "source_portal"   : "geodienste.hamburg.de — HH_WFS_Harazaen",
            "dtv"             : dtv,
        })

    print(f"✓ HaRaZäN: {len(harazaen_rows)} bike stations")

except Exception as e:
    print(f"✗ HaRaZäN fetch failed: {e}")
    print("  → Will continue with Verkehrszählstellen WFS only")

# ── Source B: HH_WFS_Verkehrszaehlstellen — all-modes station registry ────────
# Fixes applied vs original broken URL:
#   VERSION=1.1.0  (not 2.0.0)
#   typename=      (not TYPENAMES=)
#   maxFeatures=   (not COUNT=)
#   No outputFormat=application/json (returns GML, parsed below)

VZS_BASE = "https://geodienste.hamburg.de/HH_WFS_Verkehrszaehlstellen"
vzs_rows = []

print("\nFetching HH_WFS_Verkehrszaehlstellen (radverkehr_zaehlstellen layer) …")
try:
    root_vzs = wfs_fetch_gml(VZS_BASE, "app:radverkehr_zaehlstellen")
    members_vzs = [
        child for child in root_vzs.iter()
        if child.tag.split("}")[-1] == "radverkehr_zaehlstellen"
    ]
    print(f"  Raw feature elements found: {len(members_vzs)}")

    for feat in members_vzs:
        raw_lon, raw_lat = get_gml_coord(feat)
        lon, lat = raw_lon, raw_lat
        if raw_lon is not None and abs(raw_lon) > 100:
            lon, lat = utm32n_to_wgs84(raw_lon, raw_lat)

        name = get_gml_text(feat,
            "lagebezeichnung", "strassenname", "bezeichnung", "name"
        )
        station_id = get_gml_text(feat, "kennummer", "id", "zaehlstellennummer")
        last_count = get_gml_text(feat,
            "datum_letzte_zaehlungen", "letzte_zaehlung", "datum"
        )

        vzs_rows.append({
            "station_id"      : station_id,
            "datastream_id"   : "",
            "name"            : name,
            "datastream_name" : "",
            "description"     : name,
            "longitude_wgs84" : lon,
            "latitude_wgs84"  : lat,
            "provider"        : "Behörde für Verkehr und Mobilitätswende (BVM)",
            "station_type"    : "Radverkehr Zählstelle — WFS registry (mixed permanent + manual)",
            "service_name"    : "HH_WFS_Verkehrszaehlstellen",
            "layer_name"      : "app:radverkehr_zaehlstellen",
            "api_source_url"  : f"{VZS_BASE}?SERVICE=WFS&VERSION=1.1.0&REQUEST=GetFeature&typename=app:radverkehr_zaehlstellen",
            "observations_url": "",
            "mqtt_topic"      : "",
            "datetime_start"  : None,
            "datetime_end"    : iso_to_dt(last_count),
            "unit"            : "counts (DTV/DTVw annual averages)",
            "observed_property": "Radverkehrsstärke",
            "sensor_type"     : "mixed — infrared / inductive / manual",
            "update_interval" : "Annual DTV; last count date per station",
            "source_portal"   : "geodienste.hamburg.de — HH_WFS_Verkehrszaehlstellen",
            "dtv"             : "",
        })

    print(f"✓ Verkehrszählstellen: {len(vzs_rows)} rad stations")

except Exception as e:
    print(f"✗ Verkehrszählstellen fetch failed: {e}")

# Combine both WFS sources
df_wfs = pd.DataFrame(harazaen_rows + vzs_rows)

# Print actual property keys found in each source for debugging
def tag_set(root, n=5):
    tags = set()
    for i, child in enumerate(root.iter()):
        if i > 500: break
        tags.add(child.tag.split("}")[-1])
    return sorted(tags)

print(f"\n✓ Combined WFS rows: {len(df_wfs)}")
if harazaen_rows:
    print(f"  HaRaZäN tags: {tag_set(root_hr)[:15]}")
if vzs_rows:
    print(f"  VZS tags    : {tag_set(root_vzs)[:15]}")
df_wfs[["name","station_type","longitude_wgs84","latitude_wgs84"]].head(5)

## 4 · Merge & Export CSV

In [ ]:
# Combine all three sources
df_all = pd.concat([df_rad, df_sr, df_wfs], ignore_index=True)

# Drop rows with no coordinates (can't place on map)
df_all = df_all.dropna(subset=["longitude_wgs84", "latitude_wgs84"])

# Consistent column order for the CSV
COL_ORDER = [
    "station_id", "datastream_id", "name", "datastream_name", "description",
    "longitude_wgs84", "latitude_wgs84",
    "provider", "station_type", "service_name", "layer_name",
    "api_source_url", "observations_url", "mqtt_topic",
    "datetime_start", "datetime_end",
    "unit", "observed_property", "sensor_type", "update_interval", "source_portal",
]
df_all = df_all.reindex(columns=COL_ORDER)

out_csv = Path("hamburg_bike_stations.csv")
df_all.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"✓ CSV saved → {out_csv.resolve()}")
print(f"  Total rows : {len(df_all)}")
print(f"  By source  :")
print(df_all["station_type"].value_counts().to_string())
df_all.head()

## 5 · Fetch Real-Time Bike Counts for Each STA Station

In [ ]:
def fetch_latest_observation(obs_url: str, session: requests.Session) -> dict:
    """Return the most recent observation {value, phenomenon_time} or empty dict."""
    if not obs_url:
        return {}
    try:
        r = session.get(obs_url, timeout=10)
        r.raise_for_status()
        vals = r.json().get("value", [])
        if vals:
            return {
                "latest_count"     : vals[0].get("result"),
                "latest_timestamp" : vals[0].get("phenomenonTime"),
            }
    except Exception:
        pass
    return {}

print("Fetching latest observations for STA stations (this may take 1-3 min) …")
sess = requests.Session()

# Only fetch for STA sources (rows with non-empty observations_url)
sta_mask = df_all["observations_url"].str.startswith(STA_BASE, na=False)
df_sta   = df_all[sta_mask].copy()

latest_counts = []
total = len(df_sta)
for i, (idx, row) in enumerate(df_sta.iterrows()):
    obs = fetch_latest_observation(row["observations_url"], sess)
    latest_counts.append({"_idx": idx, **obs})
    if (i + 1) % 20 == 0 or i + 1 == total:
        print(f"  {i+1}/{total} done", end="\r")
    time.sleep(0.05)  # gentle rate limit

print(f"\n✓ Fetched {total} latest observations")
df_obs = pd.DataFrame(latest_counts).set_index("_idx")
df_all = df_all.join(df_obs)

In [ ]:
# Save enriched CSV with live count column
df_all.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"✓ CSV updated with live counts → {out_csv.resolve()}")
print("\nLive count sample:")
df_all[["name","station_type","latest_count","latest_timestamp"]].dropna(subset=["latest_count"]).head(10)

## 6 · Kepler.gl Map — All Stations with Real-Time Counts

In [ ]:
# Build GeoJSON for the Kepler map
# We create two separate GeoJSON FeatureCollections:
#   • rad_fc   — permanent infrared count stations (circle, coloured by count)
#   • sr_fc    — StadtRAD docking stations (square, coloured by available bikes)
#   • wfs_fc   — WFS registry stations (triangle, metadata only)

def row_to_feature(row):
    if pd.isna(row.get("longitude_wgs84")) or pd.isna(row.get("latitude_wgs84")):
        return None
    props = {
        "station_id"     : str(row.get("station_id", "")),
        "name"           : str(row.get("name", "")),
        "station_type"   : str(row.get("station_type", "")),
        "provider"       : str(row.get("provider", "")),
        "sensor_type"    : str(row.get("sensor_type", "")),
        "update_interval": str(row.get("update_interval", "")),
        "unit"           : str(row.get("unit", "")),
        "api_source_url" : str(row.get("api_source_url", "")),
        "latest_count"   : float(row["latest_count"]) if pd.notna(row.get("latest_count")) else None,
        "latest_timestamp": str(row.get("latest_timestamp", "") or ""),
        "datetime_start" : str(row.get("datetime_start", "") or ""),
        "datetime_end"   : str(row.get("datetime_end", "") or ""),
    }
    return {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row["longitude_wgs84"]), float(row["latitude_wgs84"])]
        },
        "properties": props
    }

def build_fc(df_subset):
    feats = [f for f in (row_to_feature(r) for _, r in df_subset.iterrows()) if f]
    return {"type": "FeatureCollection", "features": feats}

mask_rad = df_all["service_name"] == "HH_STA_AutomatisierteVerkehrsmengenerfassung"
mask_sr  = df_all["service_name"] == "HH_STA_StadtRad"
mask_wfs = df_all["service_name"].isin(["HH_WFS_Harazaen", "HH_WFS_Verkehrszaehlstellen"])

rad_fc = build_fc(df_all[mask_rad])
sr_fc  = build_fc(df_all[mask_sr])
wfs_fc = build_fc(df_all[mask_wfs])

print(f"✓ GeoJSON built: {len(rad_fc['features'])} Radverkehr, {len(sr_fc['features'])} StadtRAD, {len(wfs_fc['features'])} WFS")

In [ ]:
# ── Kepler.gl config: three layers, each with distinct colour ramp ───────────

KEPLER_CONFIG = {
    "version": "v1",
    "config": {
        "visState": {
            "filters": [],
            "layers": [
                {
                    "id": "rad_layer",
                    "type": "point",
                    "config": {
                        "dataId": "Radverkehr Zählstellen",
                        "label": "Permanent bike counters (infrared)",
                        "color": [255, 153, 31],
                        "columns": {"lat": "latitude_wgs84", "lng": "longitude_wgs84", "altitude": None},
                        "isVisible": True,
                        "visConfig": {
                            "radius": 12,
                            "fixedRadius": False,
                            "opacity": 0.85,
                            "outline": True,
                            "thickness": 2,
                            "strokeColor": [255, 255, 255],
                            "colorRange": {
                                "name": "Uber Viz Diverging",
                                "type": "diverging",
                                "category": "Uber",
                                "colors": ["#00939C","#6EB5B8","#B3DBDD","#F0C5AA","#EB7B4D","#C94034"]
                            },
                            "radiusRange": [6, 28],
                        },
                        "colorField": {"name": "latest_count", "type": "real"},
                        "colorScale": "quantile",
                        "sizeField":  {"name": "latest_count", "type": "real"},
                        "sizeScale": "sqrt",
                        "textLabel": [{
                            "field": {"name": "name", "type": "string"},
                            "color": [255, 255, 255],
                            "size": 11,
                            "offset": [0, 14],
                            "anchor": "middle",
                            "alignment": "center",
                        }],
                    },
                },
                {
                    "id": "sr_layer",
                    "type": "point",
                    "config": {
                        "dataId": "StadtRAD Stationen",
                        "label": "StadtRAD docking stations (available bikes)",
                        "color": [72, 162, 227],
                        "columns": {"lat": "latitude_wgs84", "lng": "longitude_wgs84", "altitude": None},
                        "isVisible": True,
                        "visConfig": {
                            "radius": 8,
                            "fixedRadius": False,
                            "opacity": 0.75,
                            "outline": True,
                            "thickness": 1.5,
                            "strokeColor": [220, 240, 255],
                            "colorRange": {
                                "name": "ColorBrewer YlGnBu-6",
                                "type": "sequential",
                                "category": "ColorBrewer",
                                "colors": ["#ffffd9","#c7e9b4","#7fcdbb","#41b6c4","#1d91c0","#225ea8"]
                            },
                            "radiusRange": [5, 18],
                        },
                        "colorField": {"name": "latest_count", "type": "real"},
                        "colorScale": "quantile",
                        "sizeField":  {"name": "latest_count", "type": "real"},
                        "sizeScale": "linear",
                    },
                },
                {
                    "id": "wfs_layer",
                    "type": "point",
                    "config": {
                        "dataId": "WFS Zählstellen",
                        "label": "WFS registry stations (all modes)",
                        "color": [180, 180, 180],
                        "columns": {"lat": "latitude_wgs84", "lng": "longitude_wgs84", "altitude": None},
                        "isVisible": True,
                        "visConfig": {
                            "radius": 5,
                            "fixedRadius": True,
                            "opacity": 0.5,
                            "outline": False,
                        },
                    },
                },
            ],
            "interactionConfig": {
                "tooltip": {
                    "enabled": True,
                    "compareMode": False,
                    "fieldsToShow": {
                        "Radverkehr Zählstellen": [
                            {"name": "name", "format": None},
                            {"name": "latest_count", "format": None},
                            {"name": "latest_timestamp", "format": None},
                            {"name": "sensor_type", "format": None},
                            {"name": "update_interval", "format": None},
                            {"name": "api_source_url", "format": None},
                        ],
                        "StadtRAD Stationen": [
                            {"name": "name", "format": None},
                            {"name": "latest_count", "format": None},
                            {"name": "latest_timestamp", "format": None},
                            {"name": "provider", "format": None},
                            {"name": "api_source_url", "format": None},
                        ],
                        "WFS Zählstellen": [
                            {"name": "name", "format": None},
                            {"name": "sensor_type", "format": None},
                            {"name": "datetime_end", "format": None},
                        ]
                    }
                },
                "brush": {"size": 0.5, "enabled": False},
            },
        },
        "mapState": {
            "bearing": 0,
            "latitude": 53.560,
            "longitude": 10.005,
            "pitch": 0,
            "zoom": 11,
        },
        "mapStyle": {
            "styleType": "dark",
            "topLayerGroups": {},
            "visibleLayerGroups": {
                "label": True, "road": True, "border": False,
                "building": True, "water": True, "land": True
            },
        },
    },
}

print("✓ Kepler config defined")

In [ ]:
# ── Build and render Kepler.gl map ───────────────────────────────────────────
from keplergl import KeplerGl

# Flatten GeoJSON properties to flat DataFrames for Kepler
def geojson_to_df(fc):
    rows = []
    for feat in fc["features"]:
        row = dict(feat["properties"])
        row["longitude_wgs84"] = feat["geometry"]["coordinates"][0]
        row["latitude_wgs84"]  = feat["geometry"]["coordinates"][1]
        rows.append(row)
    return pd.DataFrame(rows)

df_rad_kgl = geojson_to_df(rad_fc)
df_sr_kgl  = geojson_to_df(sr_fc)
df_wfs_kgl = geojson_to_df(wfs_fc)

m = KeplerGl(height=700, config=KEPLER_CONFIG)
m.add_data(data=df_rad_kgl, name="Radverkehr Zählstellen")
m.add_data(data=df_sr_kgl,  name="StadtRAD Stationen")
m.add_data(data=df_wfs_kgl, name="WFS Zählstellen")

m  # displays inline in Jupyter

In [ ]:
# Save to standalone HTML file
html_out = Path("hamburg_bike_map.html")
m.save_to_html(file_name=str(html_out), read_only=False)
print(f"✓ Kepler map saved → {html_out.resolve()}")

## 7 · Quick Summary Statistics

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Hamburg Bike Counting Infrastructure — Station Overview", fontsize=13, fontweight="bold")

# ── Panel 1: station count by type ──────────────────────────────────────────
type_counts = df_all["station_type"].value_counts()
colors = ["#FF9920", "#48A2E3", "#aaaaaa"]
axes[0].barh(range(len(type_counts)), type_counts.values, color=colors[:len(type_counts)])
axes[0].set_yticks(range(len(type_counts)))
axes[0].set_yticklabels([t[:35] for t in type_counts.index], fontsize=9)
axes[0].set_xlabel("Number of stations")
axes[0].set_title("Stations by type")

# ── Panel 2: distribution of latest bike counts (Radverkehr only) ────────────
rad_counts = df_all.loc[mask_rad, "latest_count"].dropna()
if len(rad_counts):
    axes[1].hist(rad_counts, bins=20, color="#FF9920", edgecolor="white", linewidth=0.5)
    axes[1].set_xlabel("Latest 15-min count (bikes)")
    axes[1].set_ylabel("Stations")
    axes[1].set_title("Distribution of current bike counts\n(permanent infrared stations)")
else:
    axes[1].text(0.5, 0.5, "No live data available", ha="center", va="center")
    axes[1].set_title("Current bike counts (infrared stations)")

# ── Panel 3: StadtRAD available bikes distribution ──────────────────────────
sr_counts = df_all.loc[mask_sr, "latest_count"].dropna()
if len(sr_counts):
    axes[2].hist(sr_counts, bins=15, color="#48A2E3", edgecolor="white", linewidth=0.5)
    axes[2].set_xlabel("Available bikes")
    axes[2].set_ylabel("Docking stations")
    axes[2].set_title("StadtRAD dock availability\n(live)")
else:
    axes[2].text(0.5, 0.5, "No live data available", ha="center", va="center")
    axes[2].set_title("StadtRAD availability")

plt.tight_layout()
plt.savefig("hamburg_bike_stations_stats.png", dpi=140, bbox_inches="tight")
plt.show()
print("✓ Stats chart saved → hamburg_bike_stations_stats.png")

## 8 · MQTT Real-Time Streaming (optional — runs indefinitely)

In [ ]:
# ── Stream live 5-min observations via MQTT (run this cell in a separate terminal) ──
# This cell is NOT auto-executed. Run manually to start a live feed.

# pip install paho-mqtt

DEMO_CELL = '''
import paho.mqtt.client as mqtt, json, pandas as pd

# Pick up to N station datastream IDs from your CSV
df = pd.read_csv("hamburg_bike_stations.csv")
rad = df[df["service_name"]=="HH_STA_AutomatisierteVerkehrsmengenerfassung"].dropna(subset=["datastream_id"])
ds_ids = rad["datastream_id"].astype(str).unique()[:10]   # first 10 stations

def on_connect(client, userdata, flags, rc):
    print(f"Connected (rc={rc}). Subscribing to {len(ds_ids)} streams …")
    for ds_id in ds_ids:
        topic = f"v1.1/Datastreams({ds_id})/Observations"
        client.subscribe(topic)
        print(f"  subscribed: {topic}")

def on_message(client, userdata, msg):
    payload = json.loads(msg.payload.decode())
    ds_id = msg.topic.split("(")[1].split(")")[0]
    name  = rad.loc[rad["datastream_id"]==ds_id, "name"].values
    label = name[0] if len(name) else ds_id
    print(f"{payload.get(\'phenomenonTime\','--'):32s}  {label:40s}  count={payload.get(\'result\')}")

c = mqtt.Client()
c.on_connect = on_connect
c.on_message = on_message
c.connect("iot.hamburg.de", 1883, 60)
c.loop_forever()   # Ctrl-C to stop
'''

print("MQTT streaming snippet (copy and run separately):")
print(DEMO_CELL)

---
## Output files

| File | Contents |
|------|----------|
| `hamburg_bike_stations.csv` | Full station inventory with coordinates, metadata and latest live counts |
| `hamburg_bike_map.html` | Kepler.gl standalone HTML map (open in any browser) |
| `hamburg_bike_stations_stats.png` | Summary bar chart + count distributions |

## CSV column reference

| Column | Description |
|--------|-------------|
| `station_id` | Unique ID in the source system (STA Thing ID or WFS Kennummer) |
| `datastream_id` | STA Datastream ID (empty for WFS) |
| `name` | Human-readable station name / location label |
| `longitude_wgs84` / `latitude_wgs84` | WGS84 decimal coordinates |
| `provider` | Data owner |
| `station_type` | Sensor technology and measurement mode |
| `api_source_url` | Direct REST URL to the full station record |
| `observations_url` | REST URL to fetch latest observation |
| `mqtt_topic` | MQTT topic for real-time push subscription |
| `datetime_start` / `datetime_end` | Coverage period from STA phenomenonTime |
| `unit` | Unit of measurement (counts, bikes, …) |
| `latest_count` | Most recent value at time of notebook run |
| `latest_timestamp` | Phenomenon time of the latest observation |